# 04 — Two-Tower Model

**CineMind Phase 1 · Step 4** — Train the centrepiece model that produces
personalised collaborative vectors and beats the RBM baseline.

### Architecture

Two small networks ("towers"):

| Tower | Input | Output |
|---|---|---|
| User tower | 64-dim ID embedding | 64-dim unit vector |
| Item tower | 64-dim ID embedding + 19 genre flags | 64-dim unit vector |

Dot product of user and item vectors = match score.

### Key tricks

| Trick | Why it matters |
|---|---|
| In-batch negatives | 512 pairs → 512×512 comparison matrix at no extra cost |
| Temperature 0.07 | Sharpens similarity distribution |
| **logQ correction** | Subtracts `log P(item)` from logits — **without this, the model loses to a popularity baseline** |
| Temporal split | Train on past, test on future — prevents data leakage |

**Requires:** `data/train_positives.csv`, `data/test_positives.csv`, `data/items.csv`

**Outputs:** `artifacts/two_tower.pt` · `artifacts/item_vecs.npy`

In [1]:
import os, sys

# Run from project root regardless of where Jupyter was launched
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Make src/ importable
if not any('src' in p for p in sys.path):
    sys.path.insert(0, 'src')

import cinemind_utils as cu
print("cwd:", os.getcwd())

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
cu.ensure_dirs()

cwd: c:\Users\shaur\Desktop\My codes shaurya\Content Recommendation system\cinemind _phase1\cinemind_phase1


## 1 · Model definition

In [2]:
class Tower(nn.Module):
    """2-layer MLP that outputs a unit-length vector."""
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, out_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, genre_t, item_counts, id_dim=64, out_dim=64):
        super().__init__()
        self.user_emb   = nn.Embedding(n_users, id_dim)
        self.item_emb   = nn.Embedding(n_items, id_dim)
        self.user_tower = Tower(id_dim, out_dim)
        self.item_tower = Tower(id_dim + 19, out_dim)   # ID + 19 genre flags
        self.temp       = 0.07
        self.register_buffer("genre_t", genre_t)
        # logQ correction: log of each item's training frequency
        logq = torch.log(torch.tensor(item_counts / item_counts.sum() + 1e-9))
        self.register_buffer("logq", logq.float())

    def user_vec(self, u):
        return self.user_tower(self.user_emb(u))

    def item_vec(self, i):
        return self.item_tower(torch.cat([self.item_emb(i), self.genre_t[i]], dim=-1))

    def forward(self, u, i):
        uv, iv = self.user_vec(u), self.item_vec(i)
        logits = uv @ iv.t() / self.temp   # [B, B] similarity matrix
        logits = logits - self.logq[i]     # logQ correction — do not remove
        labels = torch.arange(len(u))      # diagonal = true pairs
        return F.cross_entropy(logits, labels)

## 2 · Load data and build tensors

In [3]:
train = pd.read_csv("data/train_positives.csv")
test  = pd.read_csv("data/test_positives.csv")
items = pd.read_csv("data/items.csv")

n_items = int(max(train.movie_id.max(), test.movie_id.max(), items.movie_id.max())) + 1
n_users = int(max(train.user_id.max(),  test.user_id.max())) + 1

# Genre features for the item tower
genre_mat = np.zeros((n_items, 19), dtype=np.float32)
genre_mat[items.movie_id.values] = items[[f"g{i}" for i in range(19)]].to_numpy(dtype=np.float32)
genre_t = torch.tensor(genre_mat)

# Item popularity counts for logQ
item_counts = np.bincount(train.movie_id.values, minlength=n_items).astype(np.float64) + 1.0

train_u = torch.tensor(train.user_id.values)
train_i = torch.tensor(train.movie_id.values)
print(f"n_users={n_users}  n_items={n_items}  train pairs={len(train):,}")

n_users=944  n_items=1683  train pairs=44,679


## 3 · Train — 60 epochs

In [4]:
model = TwoTower(n_users, n_items, genre_t, item_counts)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)

B   = 512
idx = np.arange(len(train))

for epoch in range(60):
    np.random.shuffle(idx)
    total = 0.0
    for s in range(0, len(idx) - B, B):
        b    = idx[s:s + B]
        loss = model(train_u[b], train_i[b])
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"  epoch {epoch+1:3d}  loss {total / (len(idx) // B):.4f}")

  epoch  10  loss 5.7588
  epoch  20  loss 5.4742
  epoch  30  loss 5.3409
  epoch  40  loss 5.2533
  epoch  50  loss 5.1859
  epoch  60  loss 5.1383


## 4 · Evaluate — three rankers on the same test set

In [5]:
model.eval()
with torch.no_grad():
    item_vecs = model.item_vec(torch.arange(n_items))   # precompute once

seen    = train.groupby("user_id").movie_id.apply(set).to_dict()
truth   = test.groupby("user_id").movie_id.apply(set).to_dict()
top_pop = train.movie_id.value_counts().index.to_numpy()

pop_cnt = train.movie_id.value_counts().reindex(range(n_items), fill_value=0).to_numpy()
log_pop = np.log1p(pop_cnt); log_pop /= log_pop.max()

def pop_rank(u, k=10):
    s = seen.get(u, set())
    return [m for m in top_pop if m not in s][:k]

def tt_rank(u, k=10):
    with torch.no_grad():
        scores = (model.user_vec(torch.tensor([u])) @ item_vecs.t()).squeeze(0).numpy()
    for m in seen.get(u, set()):
        scores[m] = -1e9
    return np.argpartition(-scores, k)[:k]

def hybrid_rank(u, k=10, alpha=0.30):
    with torch.no_grad():
        scores = (model.user_vec(torch.tensor([u])) @ item_vecs.t()).squeeze(0).numpy()
    scores += alpha * log_pop
    for m in seen.get(u, set()):
        scores[m] = -1e9
    return np.argpartition(-scores, k)[:k]

p_pop, r_pop = cu.precision_recall_at_k(pop_rank, truth)
p_tt,  r_tt  = cu.precision_recall_at_k(tt_rank,  truth)
p_hy,  r_hy  = cu.precision_recall_at_k(hybrid_rank, truth)

print(f"\n{'Model':<26}{'Precision@10':>14}{'Recall@10':>14}")
print("-" * 56)
print(f"{'Popularity baseline':<26}{p_pop:>14.4f}{r_pop:>14.4f}")
print(f"{'Two-tower':<26}{p_tt:>14.4f}{r_tt:>14.4f}")
print(f"{'Two-tower + pop prior':<26}{p_hy:>14.4f}{r_hy:>14.4f}")
print("-" * 56)


Model                       Precision@10     Recall@10
--------------------------------------------------------
Popularity baseline               0.0603        0.0665
Two-tower                         0.0581        0.0917
Two-tower + pop prior             0.0945        0.1254
--------------------------------------------------------


## 5 · Save artifacts

In [6]:
torch.save(model.state_dict(), os.path.join(cu.ARTIFACT_DIR, "two_tower.pt"))
np.save(os.path.join(cu.ARTIFACT_DIR, "item_vecs.npy"), item_vecs.numpy())
print("Saved artifacts/two_tower.pt")
print("Saved artifacts/item_vecs.npy")
print("Next -> 05_qdrant_indexing.ipynb")

Saved artifacts/two_tower.pt
Saved artifacts/item_vecs.npy
Next -> 05_qdrant_indexing.ipynb
